# Módulo 2 — Movimiento Browniano Geométrico

**Proyecto:** Métodos de Monte Carlo en Finanzas Cuantitativas  
**Autor:** Jorge Rodríguez Lázaro  
**Última actualización:** 2026-07-05

---

## Objetivo del módulo

Este módulo construye el modelo estándar para el precio de un activo financiero: el **Movimiento Browniano Geométrico (GBM)**. Partimos de las dos piezas del Módulo 1 — la simulación Monte Carlo y el movimiento browniano — y las aplicamos al problema de modelar precios reales.

El objetivo es triple:

1. Entender de dónde sale la fórmula del GBM y por qué es el modelo natural para precios.
2. Simular trayectorias de precios y visualizar su comportamiento.
3. Estimar los parámetros del modelo ($\mu$ y $\sigma$) a partir de datos reales de mercado.

## Estructura

| Sección | Tema | Conexión con el proyecto |
|---------|------|--------------------------|
| 1 | Del paseo aleatorio al GBM | Por qué modelamos logaritmos y de dónde sale la fórmula. |
| 2 | Simulación de trayectorias | Discretización de Euler-Maruyama e implementación vectorizada. |
| 3 | Estimación de parámetros | Calibración de μ y σ con datos reales de Yahoo Finance. |
| 4 | Validación | Comparación entre trayectorias simuladas y precio histórico real. |

## Referencias

- Glasserman, P. (2003). *Monte Carlo Methods in Financial Engineering*, cap. 3.
- Hull, J. C. (2018). *Options, Futures and Other Derivatives*, cap. 15.
- Shreve, S. E. (2004). *Stochastic Calculus for Finance II*, cap. 4.

---

# Sección 1 — Del paseo aleatorio al GBM

## 1.1 Por qué no vale el movimiento browniano directo

El movimiento browniano $W_t$ tiene dos problemas como modelo de precios:

1. **Puede ser negativo.** $W_t \in \mathbb{R}$ para todo $t$, pero un precio $S_t > 0$ siempre.
2. **Los cambios son absolutos, no proporcionales.** En finanzas lo relevante es el cambio porcentual: una subida de 1€ en una acción de 10€ (10%) no es lo mismo que en una de 1000€ (0.1%). Un modelo aditivo no captura esto.

## 1.2 La solución: modelar las rentabilidades logarítmicas

Definimos la **rentabilidad logarítmica** entre dos instantes como:

$$
r_t = \log\frac{S_t}{S_{t-1}}
$$

Esta cantidad puede ser positiva o negativa (el precio sube o baja), no está acotada, y es **proporcional** al nivel del precio por construcción. Es el objeto natural a modelar.

El supuesto del GBM es que las rentabilidades logarítmicas son i.i.d. y normales:

$$
r_t \sim \mathcal{N}\!\left(\left(\mu - \frac{\sigma^2}{2}\right)\Delta t,\; \sigma^2 \Delta t\right)
$$

donde $\mu$ es la **rentabilidad esperada** (drift) y $\sigma$ es la **volatilidad**.

## 1.3 De las rentabilidades al precio: la fórmula del GBM

Acumulando las rentabilidades logarítmicas desde $t=0$ hasta $t=T$:

$$
\log\frac{S_T}{S_0} = \sum_{i=1}^{N} r_i \sim \mathcal{N}\!\left(\left(\mu - \frac{\sigma^2}{2}\right)T,\; \sigma^2 T\right)
$$

Despejando $S_T$:

$$
\boxed{S_T = S_0 \, \exp\!\left(\left(\mu - \frac{\sigma^2}{2}\right)T + \sigma W_T\right)}
$$

donde $W_T \sim \mathcal{N}(0, T)$ es el movimiento browniano en el instante $T$.

## 1.4 La corrección $\sigma^2/2$ — corrección de Jensen

¿Por qué aparece $\mu - \sigma^2/2$ y no simplemente $\mu$?

Porque la esperanza de $e^X$ no es $e^{\mathbb{E}[X]}$ cuando $X$ es aleatoria. Si $X \sim \mathcal{N}(m, v)$:

$$
\mathbb{E}[e^X] = e^{m + v/2}
$$

Queremos que $\mathbb{E}[S_T] = S_0 \, e^{\mu T}$ — el precio crece en promedio a tasa $\mu$. Si ponemos $m = (\mu - \sigma^2/2)T$ y $v = \sigma^2 T$:

$$
\mathbb{E}[S_T] = S_0 \cdot e^{(\mu - \sigma^2/2)T + \sigma^2 T/2} = S_0 \, e^{\mu T} \checkmark
$$

El término $-\sigma^2/2$ compensa exactamente el exceso que introduce la exponencial, garantizando que la rentabilidad esperada del precio sea $\mu$ tal y como deseamos.